In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,MinMaxScaler,RobustScaler,label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score,f1_score,recall_score,precision_score,confusion_matrix,roc_auc_score,balanced_accuracy_score

N_SPLITS = 3
RANDOM_STATE=42
skf = StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)

data_path = Path.cwd().parent / "data" / "playground-series-s6e6"
data_orig_path = Path.cwd().parent / "data" / "star_classification.csv"

df_train = pd.read_csv(data_path / "train.csv")
df_test = pd.read_csv(data_path / "test.csv")
data_orig = pd.read_csv(data_orig_path)

df_exp = df_train.copy()
df_test_exp = df_test.copy()

train_ids = df_exp["id"].values
test_ids = df_test_exp["id"].values

df_exp.drop(["id"],axis=1,inplace=True)
X_test = df_test_exp.drop(["id"],axis=1)

X_train,y_train = df_exp.drop(["class"],axis=1), df_exp["class"]

target_col = ["class"]
categorical_cols = ["spectral_type","galaxy_population"]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols + target_col ]
objects_for_test = {}

In [5]:

def create_astronomical_bins(df:pd.Dataframe):
    if "spectral_type" not in df.columns:
        df["spectral_type"] = pd.cut(df["r"] - df["g"],bins=[-np.inf,-1,-0.5,0,np.inf],labels=["M","G/K","A/F","O/B"]).astype("str")
    if "galaxy_population" not in df.columns:
        df["galaxy_population"] = pd.cut(df["u"] - df["r"],bins=[-np.inf,2.2,np.inf],labels=["Blue_Cloud","Red_Sequence"]).astype("str")
    if "combo" not in df.columns:
        df["combo"] = df["spectral_type"] + "_" + df["galaxy_population"]

    return df

def generate_target_encodings(train_df:pd.DataFrame,test_df:pd.DataFrame,orig_df:pd.DataFrame):
    orig_df = create_astronomical_bins(orig_df)
    target_map = {"GALAXY":0,"QSO":1,"STAR":2}
    y_orig = orig_df["class"].map(target_map)

    spectral_map = y_orig.groupby(orig_df["spectral_type"]).mean()
    galaxypop_map = y_orig.groupby(orig_df["galaxy_population"]).mean()
    combo_map = y_orig.groupby(orig_df["combo"]).mean()


    for df in [train_df,test_df]:
        df = create_astronomical_bins(df)
        df["orig_spectral_target"] = df["spectral_type"].map(spectral_map) 
        df["orig_galaxypop_target"] =df["galaxy_population"].map(galaxypop_map)
        df["orig_combo_target"] = df["combo"].map(combo_map)

    return train_df,test_df


X_train,X_test = generate_target_encodings(train_df=X_train,test_df=X_test,orig_df=data_orig)

In [8]:
X_train

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,combo,orig_spectral_target,orig_galaxypop_target,orig_combo_target
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,O/B_Blue_Cloud,1.131426,0.905193,1.134708
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577342,223.539288,2.503680,20.828729,18.854201,17.703108,17.190536,16.551356,0.511524,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412
577343,223.895970,40.769343,23.734743,22.359173,20.697865,19.180264,18.947275,0.658589,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412
577344,52.258927,0.671887,21.944250,21.215856,19.025966,18.772276,18.203397,0.376342,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412
577345,247.362248,50.659819,21.969881,21.622766,20.987575,20.930924,21.478134,2.868359,G/K,Blue_Cloud,G/K_Blue_Cloud,0.635138,0.905193,0.649309


In [35]:
# def redshift_correction(df:pd.DataFrame):
df = X_train.copy()

bands = ["u","g","r","i","z"]

lambda_obs = np.array([355.1,468.6,616.5,748.1,893.1]) # color filters
lambda_obs = np.broadcast_to(lambda_obs,(len(df),lambda_obs.shape[0])) # n_rows,5

observed_mags = np.array(df[bands].values,dtype=float) # n_rows,5; Y
redshifts = np.array(df["redshift"].abs().values,dtype=float)[:,None] # n_rows,1

lambda_emit = lambda_obs / (1+redshifts) # X


idx = np.sum(lambda_emit[:,:,None] <= lambda_obs[:,None,:],axis=1) -1 # numpy broadcast to n_rows,5,5
idx = np.clip(idx,0,lambda_emit.shape[1]-2)

x0,x1 = np.take_along_axis(lambda_emit,idx,axis=1), np.take_along_axis(lambda_emit,idx+1,axis=1)
y0,y1 = np.take_along_axis(observed_mags,idx,axis=1), np.take_along_axis(observed_mags,idx,axis=1)

slopes = (y1-y0)/(x1-x0)
observed_mags_emitted = y0 + slopes*(lambda_obs - x0)
df[[band+"_redshift_corrected" for band in bands]] = observed_mags_emitted

df

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,combo,orig_spectral_target,orig_galaxypop_target,orig_combo_target,u_redshift_corrected,g_redshift_corrected,r_redshift_corrected,i_redshift_corrected,z_redshift_corrected
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412,21.895559,20.357926,19.257113,19.257113,19.257113
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412,20.778509,19.087062,17.587208,17.226067,17.226067
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,O/B_Blue_Cloud,1.131426,0.905193,1.134708,20.582629,20.582629,20.582629,20.582629,20.582629
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412,21.050736,19.017754,18.365658,18.365658,18.365658
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412,19.471680,18.234449,17.899447,17.899447,17.899447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577342,223.539288,2.503680,20.828729,18.854201,17.703108,17.190536,16.551356,0.511524,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412,18.854201,17.703108,17.190536,17.190536,17.190536
577343,223.895970,40.769343,23.734743,22.359173,20.697865,19.180264,18.947275,0.658589,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412,22.359173,19.180264,19.180264,19.180264,19.180264
577344,52.258927,0.671887,21.944250,21.215856,19.025966,18.772276,18.203397,0.376342,M,Red_Sequence,M_Red_Sequence,0.257720,0.365926,0.263412,21.215856,19.025966,18.772276,18.772276,18.772276
577345,247.362248,50.659819,21.969881,21.622766,20.987575,20.930924,21.478134,2.868359,G/K,Blue_Cloud,G/K_Blue_Cloud,0.635138,0.905193,0.649309,20.930924,20.930924,20.930924,20.930924,20.930924
